# Fase 2 — Topic Modeling y análisis de sentimiento sobre el texto de las peticiones

**Autor:** Gerónimo Daguerre
**Objetivo:** identificar temas recurrentes y carga emocional en el texto libre de las peticiones (asunto + descripción), para complementar el análisis estructural de la fase 1 con causas raíz.

## ⚠️ Antes de correr este notebook

1. **Este notebook usa texto real de tickets — no lo corras sobre datos que no podés compartir sin anonimizar primero.** El CSV de entrada (`Asunto`, `Descripción`) es información interna y **no debe subirse al repo público**. Guardalo en una carpeta fuera de este repositorio (por ejemplo, en tu escritorio) y apuntá la ruta en la celda de carga de datos, más abajo.
2. **Solo se publican al repo los resultados agregados** (tópicos, palabras clave ya filtradas de nombres propios, y distribución de sentimiento) — nunca el texto original de un ticket.
3. Este notebook **no se ejecutó dentro de este entorno** porque necesita descargar modelos de embeddings (~500 MB) desde Hugging Face, y el sandbox donde armamos el repo no tiene salida a internet hacia ese dominio. Está escrito para correr en tu máquina local, donde sí vas a tener conexión completa. Instalá las dependencias de la celda siguiente y ejecutalo de punta a punta ahí.
4. Con 310 tickets (303 con texto) estás en el límite inferior de lo recomendable para BERTopic — es un volumen razonable para un proyecto de portfolio, pero esperá tópicos algo amplios, no la granularidad que tendrías con miles de documentos. Lo dejamos configurado de forma conservadora (`min_topic_size` bajo) para que igual generen agrupaciones interpretables.


In [ ]:
# Instalación (correr una sola vez en tu entorno local)
# !pip install pandas bertopic sentence-transformers pysentimiento nltk matplotlib


In [ ]:
import pandas as pd
import re
import nltk
nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords

# ==== RUTA AL CSV ORIGINAL (fuera del repo, con texto real) ====
RUTA_CSV_PRIVADO = "../../../tickets_redmine_original.csv"  # <-- ajustá esta ruta en tu máquina

df = pd.read_csv(RUTA_CSV_PRIVADO, sep=';', encoding='utf-8-sig')
df['texto'] = df['Asunto'].fillna('') + '. ' + df['Descripción'].fillna('')
df = df[df['texto'].str.len() > 15].reset_index(drop=True)
print(f"{len(df)} tickets con texto utilizable")


## 1. Preprocesamiento y anonimización del texto

Dos pasos de limpieza: el estándar (minúsculas, puntuación, stopwords) y uno propio para este caso — remover palabras con mayúscula inicial que no estén al comienzo de una oración, como heurística simple para filtrar nombres propios (personas, centros, localidades) **antes** de que cualquier palabra clave llegue a un resultado publicable.

In [ ]:
STOPWORDS_ES = set(stopwords.words('spanish'))
STOPWORDS_EXTRA = {
    'mallorca','baleares','illes','caib','inca','valencia','palma',
    'gestib','llull','ayesa','centro','curso','favor','gracias','saludos'
}
STOPWORDS_ES |= STOPWORDS_EXTRA

def quitar_nombres_propios(texto):
    oraciones = re.split(r'(?<=[.!?])\s+', texto)
    limpio = []
    for o in oraciones:
        palabras = o.split()
        filtradas = []
        for i, w in enumerate(palabras):
            core = re.sub(r'[^\wÁÉÍÓÚÑÜáéíóúñü]', '', w)
            if i > 0 and core[:1].isupper() and len(core) > 2:
                continue  # probable nombre propio / entidad
            filtradas.append(w)
        limpio.append(' '.join(filtradas))
    return ' '.join(limpio)

def preprocesar(texto):
    texto = quitar_nombres_propios(texto)
    texto = texto.lower()
    texto = re.sub(r'[^a-záéíóúñü\s]', ' ', texto)
    texto = ' '.join(w for w in texto.split() if w not in STOPWORDS_ES and len(w) > 2)
    return texto

df['texto_procesado'] = df['texto'].apply(preprocesar)
df[['texto','texto_procesado']].head(3)


## 2. Topic Modeling con BERTopic

Usamos un modelo de embeddings multilingüe (soporta español) y ajustamos `min_topic_size` hacia abajo dado el volumen acotado del dataset.

In [ ]:
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

topic_model = BERTopic(
    embedding_model=embedding_model,
    language="multilingual",
    min_topic_size=6,     # bajo, dado el volumen acotado (303 docs)
    nr_topics="auto",
    calculate_probabilities=False,
)

topics, _ = topic_model.fit_transform(df['texto_procesado'])
df['topico'] = topics

topic_info = topic_model.get_topic_info()
topic_info.head(15)


In [ ]:
# Palabras clave por tópico (ya filtradas de nombres propios en el preprocesamiento)
for t in topic_info['Topic'].head(10):
    if t == -1:
        continue  # -1 = outliers, sin tópico asignado
    palabras = [w for w, _ in topic_model.get_topic(t)][:6]
    print(f"Tópico {t} (n={topic_info.loc[topic_info.Topic==t,'Count'].values[0]}): {', '.join(palabras)}")


## 3. Análisis de sentimiento (en español)

**Importante:** no usamos VADER — su diccionario de polaridad está en inglés y da resultados sin sentido sobre texto en castellano. Usamos `pysentimiento`, un modelo BERT entrenado específicamente para español, con salida de 3 clases (POS / NEU / NEG).

In [ ]:
from pysentimiento import create_analyzer

analyzer = create_analyzer(task="sentiment", lang="es")

def sentimiento(texto):
    resultado = analyzer.predict(texto)
    return resultado.output  # 'POS', 'NEU' o 'NEG'

# Sobre el texto original (sin el filtro de nombres propios, para no perder matices de tono)
df['sentimiento'] = df['texto'].apply(sentimiento)
df['sentimiento'].value_counts(normalize=True).round(2)


## 4. Cruce tópico × sentimiento

¿Qué tópicos concentran más carga negativa? Son los candidatos a intervenir primero.

In [ ]:
cruce = pd.crosstab(df['topico'], df['sentimiento'], normalize='index').round(2)
cruce = cruce.join(topic_info.set_index('Topic')['Count']).sort_values('Count', ascending=False)
cruce.head(10)


In [ ]:
import matplotlib.pyplot as plt

NAVY, AMBER, TEAL = '#10151E', '#B8862B', '#2E6F6B'
top_topicos = topic_info[topic_info.Topic != -1].head(8)

fig, ax = plt.subplots(figsize=(8,4.5))
ax.barh(top_topicos['Name'], top_topicos['Count'], color=TEAL)
ax.set_title('Volumen por tópico detectado', fontsize=13, fontweight='bold', color=NAVY, loc='left')
for s in ['top','right']: ax.spines[s].set_visible(False)
plt.tight_layout()
plt.savefig('../../assets/06_topicos.png', transparent=True)
plt.show()


## 5. Publicar solo el resultado agregado

Guardamos un CSV con tópico, tamaño, palabras clave (ya sin nombres propios) y distribución de sentimiento — **sin ningún texto original de ticket**. Este archivo sí es seguro para subir al repo.

In [ ]:
resumen = topic_info[topic_info.Topic != -1][['Topic','Count']].copy()
resumen['palabras_clave'] = resumen['Topic'].apply(
    lambda t: ', '.join(w for w, _ in topic_model.get_topic(t)[:6])
)
resumen = resumen.merge(
    df.groupby('topico')['sentimiento'].value_counts(normalize=True).unstack().round(2),
    left_on='Topic', right_index=True, how='left'
)
resumen.to_csv('../../data/topicos_sentimiento_resumen.csv', index=False)
resumen


## 6. Lectura

Completar después de correr el notebook con los resultados reales: qué tópicos concentran más volumen, cuáles tienen mayor proporción de sentimiento negativo, y qué acción de mejora sugiere cada patrón (documentación, cambio de regla de negocio, capacitación a usuarios, etc.).

---
*Fase 1 (análisis estructural): [`../redmine_ticket_analysis.ipynb`](../redmine_ticket_analysis.ipynb) · [Case study](../../site/index.html)*
